# 11.4 — Parameter interactions

**Question.** Do prespecified historical parameter pairs have joint effects beyond additive main effects? The `test` profile validates one 2×2 pair on synthetic cells; screen/full run the config-defined grids with env-resolved features. Exact optimizer counts are printed before execution.

Runs resume from signed artifacts under the legacy sensitivity root. Heatmaps report seed-averaged means and residuals after removing both additive main effects. Residuals are descriptive within the tested grid, not causal interactions, and sparse grids or weak optimizer replication limit interpretation.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from estonia_landuse.sensitivity.analysis import (
    estimate_interaction_surface,
    select_matched_baseline_runs,
    summarize_interaction_noise,
)
from estonia_landuse.sensitivity.config import DEFAULT_SEEDS, INTERACTION_LEVELS
from estonia_landuse.sensitivity.plots import plot_interaction_heatmap
from estonia_landuse.sensitivity.runner import run_manifest
from estonia_landuse.sensitivity.sampling import INTERACTION_PAIRS, build_interaction_manifest, manifest_run_count, manifest_summary

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
HISTORICAL_ROOT = PROJECT_ROOT.parent.parent if PROJECT_ROOT.parent.name == ".worktrees" else PROJECT_ROOT
PROFILE = os.environ.get("SENSITIVITY_PROFILE", "test")
N_WORKERS = int(os.environ.get("SENSITIVITY_N_WORKERS", "2"))
OVERWRITE = os.environ.get("SENSITIVITY_OVERWRITE", "false").lower() == "true"
OUTPUT_ROOT = Path(os.environ.get("SENSITIVITY_OUTPUT_ROOT", PROJECT_ROOT / "data/processed/legacy_sensitivity")).resolve()
FEATURES_PATH = Path(os.environ.get("SENSITIVITY_FEATURES_PATH", HISTORICAL_ROOT / "data/processed/learned_carbon/features_with_forest.parquet")).resolve()
SEEDS = (0, 1) if PROFILE == "test" else DEFAULT_SEEDS[PROFILE]
SCENARIOS = ("balanced",)
PAIRS = INTERACTION_PAIRS[:1] if PROFILE == "test" else INTERACTION_PAIRS
LEVELS = INTERACTION_LEVELS[PROFILE]
OUTCOMES = ("biodiversity_gain", "carbon_gain", "cost", "changed_pct")


In [ ]:
if PROFILE == "test":
    position = np.linspace(0.0, 1.0, 12)
    context = pd.DataFrame({"cell_id": np.arange(1, 13), "forest_pct": 0.35 + 0.03 * position, "wetland_pct": 0.10 + 0.02 * position, "agriculture_pct": 0.30 - 0.03 * position, "grassland_pct": 0.15 - 0.02 * position, "urban_pct": np.full(12, 0.05), "water_pct": np.full(12, 0.05), "protected_overlap_pct": 0.05 * position, "wetland_suitability": 0.2 + 0.6 * position, "opportunity_cost_proxy": 0.1 + 0.5 * position, "predicted_tco2_ha_yr": 2.5 + 2.0 * position, "peat_overlap_pct": 0.4 * position})
    feature_columns = ["wetland_suitability", "opportunity_cost_proxy"]
else:
    if not FEATURES_PATH.exists(): raise FileNotFoundError(f"Missing historical feature input: {FEATURES_PATH}")
    context = pd.read_parquet(FEATURES_PATH)
    feature_columns = [name for name in ("urban_pct", "agriculture_pct", "grassland_pct", "forest_pct", "wetland_pct", "water_pct", "naturalness_score", "carbon_score", "protected_overlap_pct", "wetland_suitability", "biodiversity_proxy", "opportunity_cost_proxy", "rohemeeter_norm") if name in context]
    if not feature_columns: raise ValueError("No preserved Notebook 10 feature columns found")


In [ ]:
manifests = [build_interaction_manifest(profile=PROFILE, pair=pair, levels=LEVELS, scenarios=SCENARIOS, seeds=SEEDS) for pair in PAIRS]
manifest = pd.concat(manifests, ignore_index=True)
planned_runs = manifest_run_count(manifest)
manifest_summary(manifest)
display(manifest.head(6))


In [ ]:
statuses = run_manifest(context, feature_columns, manifest, OUTPUT_ROOT, PROFILE, overwrite=OVERWRITE, n_workers=min(N_WORKERS, planned_runs), progress=lambda completed, total, status: print(f"[{completed}/{total}] {status}"))
if statuses["status"].eq("failed").any(): raise RuntimeError(statuses.loc[statuses["status"].eq("failed"), ["sample_id", "seed", "error_message"]].to_string(index=False))
assert len(statuses) == planned_runs
display(statuses["status"].value_counts())


In [ ]:
metrics = pd.concat([pd.read_parquet(path) for path in statuses["metrics_path"]], ignore_index=True)
baseline_paths = sorted((OUTPUT_ROOT / "runs" / "baseline").rglob("seed_*.parquet"))
if not baseline_paths: raise FileNotFoundError("Run Notebook 11.1 with this SENSITIVITY_OUTPUT_ROOT first")
baseline_candidates = pd.concat([pd.read_parquet(path) for path in baseline_paths], ignore_index=True)
baseline = select_matched_baseline_runs(
    baseline_candidates, metrics, expected_scenarios=SCENARIOS, expected_seeds=SEEDS
)
interaction_noise = summarize_interaction_noise(metrics, baseline, OUTCOMES)
display(interaction_noise.sort_values("max_abs_residual_to_noise", ascending=False)[[
    "scenario", "parameter_x", "parameter_y", "outcome",
    "max_abs_interaction_residual", "rms_interaction_residual", "baseline_sd",
    "max_abs_residual_to_noise", "rms_residual_to_noise",
]])
for parameter_x, parameter_y in PAIRS:
    pair_metrics = metrics.loc[metrics["parameter_x"].eq(parameter_x) & metrics["parameter_y"].eq(parameter_y)]
    for outcome in OUTCOMES:
        surface = estimate_interaction_surface(pair_metrics, outcome)
        display(surface)
        for value_column in ("mean", "interaction_residual"):
            figure, _ = plot_interaction_heatmap(surface, value_column, parameter_x, parameter_y)
            display(figure)
            plt.close(figure)
